# 🌱 Plant Doctor: train the leaf models on Colab

The app uses six small models: a **router** that says which of the 5 crops a leaf is (or "not a crop I know"), and one **specialist per crop** that says healthy or which disease.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (about 1–1.5 hours)
3. The last cell downloads `plant_doctor_model.zip`. Unzip it into the repo's `app/public/` folder, so you get `app/public/models/router.onnx`, the five crop models, `manifest.json` and `app/public/refs/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 https://github.com/AadidevRaizada/PlantDisease.git
%cd PlantDisease/training
!pip -q install timm onnx onnxscript onnxruntime pyarrow opencv-python-headless

In [ ]:
# Download the raw datasets from Hugging Face (~8 GB, fast on Colab)
!sh download.sh

In [ ]:
# Build a balanced image folder (up to 1200 images per class, field photos first),
# then cut the leaf out of every plain-background lab photo so training can put it on real field backgrounds
!python prepare.py
!python masks.py --workers 2

In [ ]:
# Stage 1: the leaf-type router. Stage 2: one specialist per crop, at a higher resolution, starting from the router.
!python train.py --task router --size 224 --epochs 10 --batch 96 --lr 8e-4
for crop in ["cotton", "rice", "soybean", "sugarcane", "tomato"]:
    !python train.py --task {crop} --size 288 --epochs 14 --batch 64 --lr 5e-4 --init runs/router/best.pt
# Score the whole pipeline on held-out photos
!python evaluate.py

In [ ]:
# Export ONNX models + reference images into ../app/public, then zip and download
!python export.py
!cd ../app/public && zip -qr /content/plant_doctor_model.zip models refs
from google.colab import files
files.download('/content/plant_doctor_model.zip')